<a href="https://colab.research.google.com/github/ernanovi/end-to-end-erp-bi/blob/main/API_to_BigQuery_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import requests
import pandas as pd
import pandas_gbq
from google.colab import auth
from google.cloud import bigquery

# Configuration
API_URL = "https://dummyjson.com/products?limit=50"
PROJECT_ID = "belajar-dwh-1"
DATASET_ID = "erp_dwh"
TABLE_ID = "fact_products"
DESTINATION_TABLE = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# Authentication
auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

print("BigQuery authentication successful.")

# Extraction
def extract_products(api_url: str) -> list:
    response = requests.get(
        api_url,
        timeout=30
    )
    response.raise_for_status()

    data = response.json()
    products = data.get("products", [])

    if not products:
        raise ValueError("API returned no product data.")

    print(f"Extracted {len(products)} products.")
    return products

# Transformation
def transform_products(products: list) -> pd.DataFrame:
    df = pd.DataFrame(products)

    required_columns = [
        "id",
        "title",
        "category",
        "price",
        "stock",
        "rating"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    df = df[required_columns].copy()

    df = df.rename(columns={
        "id": "product_id",
        "title": "product_name",
        "price": "unit_price",
        "stock": "stock_quantity"
    })

    df["product_id"] = pd.to_numeric(
        df["product_id"],
        errors="coerce"
    ).astype("Int64")

    df["unit_price"] = pd.to_numeric(
        df["unit_price"],
        errors="coerce"
    )

    df["stock_quantity"] = pd.to_numeric(
        df["stock_quantity"],
        errors="coerce"
    )

    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    )

    df["inventory_value"] = (
        df["unit_price"] * df["stock_quantity"]
    )

    df = df.dropna(
        subset=[
            "product_id",
            "product_name",
            "unit_price"
        ]
    )

    return df

# Data Quality Validation
def validate_products(df: pd.DataFrame) -> None:
    if df["product_id"].isna().any():
        raise ValueError(
            "Validation failed: product_id contains NULL values."
        )

    if df["product_id"].duplicated().any():
        raise ValueError(
            "Validation failed: duplicate product_id detected."
        )

    if (df["unit_price"] < 0).any():
        raise ValueError(
            "Validation failed: negative unit_price detected."
        )

    if (df["stock_quantity"] < 0).any():
        raise ValueError(
            "Validation failed: negative stock_quantity detected."
        )

    if (df["inventory_value"] < 0).any():
        raise ValueError(
            "Validation failed: negative inventory_value detected."
        )

    print("Data quality validation passed.")

# Load to BigQuery
def load_to_bigquery(
    df: pd.DataFrame,
    destination_table: str,
    project_id: str
) -> None:
    pandas_gbq.to_gbq(
        dataframe=df,
        destination_table=destination_table,
        project_id=project_id,
        if_exists="replace"
    )

    print(
        f"Loaded {len(df)} rows into {destination_table}"
    )

# Run Pipeline
products = extract_products(API_URL)

df_products = transform_products(products)

print("\nTransformed dataset:")
display(df_products.head())

print(f"\nDataset shape: {df_products.shape}")

validate_products(df_products)

load_to_bigquery(
    df=df_products,
    destination_table=DESTINATION_TABLE,
    project_id=PROJECT_ID
)

# Verify BigQuery Load
query = f"""
SELECT
    product_id,
    product_name,
    category,
    unit_price,
    stock_quantity,
    rating,
    inventory_value
FROM `{DESTINATION_TABLE}`
ORDER BY product_id
LIMIT 10
"""

verification_df = client.query(query).to_dataframe()

print("\nBigQuery verification:")
display(verification_df)

BigQuery authentication successful.
Extracted 50 products.

Transformed dataset:


,product_id,product_name,category,unit_price,stock_quantity,rating,inventory_value
0,1,Essence Mascara Lash Princess,beauty,9.99,99,2.56,989.01
1,2,Eyeshadow Palette with Mirror,beauty,19.99,34,2.86,679.66
2,3,Powder Canister,beauty,14.99,89,4.64,1334.11
3,4,Red Lipstick,beauty,12.99,91,4.36,1182.09
4,5,Red Nail Polish,beauty,8.99,79,4.32,710.21



Dataset shape: (50, 7)
Data quality validation passed.


100%|██████████| 1/1 [00:00<00:00, 10565.00it/s]


Loaded 50 rows into belajar-dwh-1.erp_dwh.fact_products

BigQuery verification:


,product_id,product_name,category,unit_price,stock_quantity,rating,inventory_value
0,1,Essence Mascara Lash Princess,beauty,9.99,99,2.56,989.01
1,2,Eyeshadow Palette with Mirror,beauty,19.99,34,2.86,679.66
2,3,Powder Canister,beauty,14.99,89,4.64,1334.11
3,4,Red Lipstick,beauty,12.99,91,4.36,1182.09
4,5,Red Nail Polish,beauty,8.99,79,4.32,710.21
5,6,Calvin Klein CK One,fragrances,49.99,29,4.37,1449.71
6,7,Chanel Coco Noir Eau De,fragrances,129.99,58,4.26,7539.42
7,8,Dior J'adore,fragrances,89.99,98,3.80,8819.02
8,9,Dolce Shine Eau de,fragrances,69.99,4,3.96,279.96
9,10,Gucci Bloom Eau de,fragrances,79.99,91,2.74,7279.09
